# Lab 3 — A tiny transformer with causal and masked objectives

COMP-672 · John Glossner

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/glossner/COMP-672-Labs/blob/main/labs/week03_starter.ipynb)

**First: File → Save a copy in Drive.** Sign in to Google to run and save. Rename your copy, for example `COMP672_Lab03_YourName.ipynb`, and work in that copy.


## Install the lab environment

Use Colab’s **default Python CPU runtime**; you do not need to select an older Python runtime.


In [54]:
import sys
import subprocess
import importlib.metadata as metadata

if not (3, 11) <= sys.version_info[:2] <= (3, 14):
    raise RuntimeError("This lab supports Python 3.11–3.14. Reopen the latest notebook from the course README if your runtime is newer.")

print("Runtime Python:", sys.version.split()[0])

LAB_PACKAGES = {'numpy': '2.1.3', 'scipy': '1.16.3', 'scikit-learn': '1.6.1', 'matplotlib': '3.10.0'}
# Python 3.14 needs newer binary wheels; retain the established pins on 3.11–3.13.
if sys.version_info[:2] >= (3, 14):
    LAB_PACKAGES.update({'numpy': '2.3.5', 'scikit-learn': '1.7.2', 'matplotlib': '3.10.8'})
LAB_PACKAGES["torch"] = "2.11.0"
def installed_version(name):
    try:
        return metadata.version(name).split("+")[0]
    except metadata.PackageNotFoundError:
        return None

needed = [f"{name}=={version}" for name, version in LAB_PACKAGES.items()
          if name != "torch" and installed_version(name) != version]
if needed:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "--only-binary=:all:", *needed])
if installed_version("torch") != LAB_PACKAGES["torch"]:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "--only-binary=:all:",
                           "torch==2.11.0", "--index-url",
                           "https://download.pytorch.org/whl/cpu"])
print("Package setup complete. Restart the session if Colab requests it.")


Runtime Python: 3.13.18
Package setup complete. Restart the session if Colab requests it.


In [55]:
import platform
import importlib

LAB_VERSIONS = {"python": platform.python_version()}
for distribution, module_name in [("numpy", "numpy"), ("scipy", "scipy"),
                                  ("scikit-learn", "sklearn"), ("matplotlib", "matplotlib")]:
    module = importlib.import_module(module_name)
    LAB_VERSIONS[distribution] = module.__version__
import torch
torch.set_num_threads(1)
LAB_VERSIONS["torch"] = torch.__version__
for name, expected in LAB_PACKAGES.items():
    if LAB_VERSIONS[name].split("+")[0] != expected:
        raise RuntimeError(f"Restart the session: {name} is already imported at a different version.")
print(LAB_VERSIONS)
print("CPU environment ready. Synthetic data will be generated by the lab functions.")


{'python': '3.13.18', 'numpy': '2.1.3', 'scipy': '1.16.3', 'scikit-learn': '1.6.1', 'matplotlib': '3.10.0', 'torch': '2.11.0+cpu'}
CPU environment ready. Synthetic data will be generated by the lab functions.


# Lab 3 — A tiny transformer with causal and masked objectives

**2 points.** Chapters 7, 8, 9; outcomes LO3, LO8. Allow 4–5 hours including the 60-minute lab meeting. Complete the Python starter or the Jupyter starter, not both. The scaffolding contains intentional `NotImplementedError` gaps.

## Implementation tasks

1. Implement scaled attention and a causal mask; verify that changing a future value leaves earlier causal outputs unchanged.

2. Implement shifted next-token cross-entropy and explain why the last position is excluded.

3. Implement a selected-position masked loss and contrast it with the causal objective; ensure unselected positions do not contribute.

4. Implement the rank-2 adapter forward pass with a frozen base head and scale alpha/r. Count trainable parameters and check the zero-update initialization.


Functions to complete: `attention`, `causal_loss`, `masked_loss`, `LoRALinear.forward`.

## Required experiments

- Run the tiny causal pretraining and head-only LoRA adaptation experiments. Report initial loss, fixed-training-endpoint test loss, adapter parameter count, and adaptation loss before and after. Use the same adapted-distribution test set for the before/after comparison.

- Run the bidirectional masked model. Perturb right context to demonstrate visibility and explain why its loss is not directly ranked against causal perplexity.

## Brief analysis in your notebook or code comments

1. Which causal-mask bug could create deceptively low training loss?

2. What can adaptation of only an output head change, and what can it not directly change in the frozen backbone?

3. The finite grammar repeats possible sequences across independent draws. Explain why these results measure a small distribution-fitting exercise rather than held-out linguistic compositionality.


Answer each question in approximately 2–4 sentences beside the results. This is part of the lab, with no separate report.



## Starter code

Complete the exercise functions below. Keep the supplied checks and add the cases required by the handout.

In [56]:
"""Week 3 student starter: tiny transformer; no downloaded weights or corpora."""


'Week 3 student starter: tiny transformer; no downloaded weights or corpora.'

In [57]:
import math


In [58]:
import torch


In [59]:
from torch import nn


In [60]:
from torch.nn import functional as F


In [61]:
torch.set_num_threads(1)


In [62]:

def attention(q, k, v, causal=True):
    """Inputs B,H,T,d. Return attended values and row-normalized weights."""
    import math
    import torch

    # Step 1: Calculate scaled attention scores
    d = q.size(-1)
    scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(d)

    # Step 2: Apply causal mask to block future tokens
    if causal:
        T = q.size(-2)
        S = k.size(-2)
        mask = torch.triu(
            torch.ones(T, S, device=q.device, dtype=torch.bool),
            diagonal=1
        )
        scores = scores.masked_fill(mask, float("-inf"))

    # Step 3: Normalize attention weights
    weights = torch.softmax(scores, dim=-1)

    # Step 4: Calculate attended values
    output = torch.matmul(weights, v)

    return output, weights


In [63]:

import torch

torch.manual_seed(42)

q = torch.randn(2, 2, 4, 8)
k = torch.randn(2, 2, 4, 8)
v = torch.randn(2, 2, 4, 8)

output, weights = attention(q, k, v, causal=True)

print("Output shape:", output.shape)
print("Weights shape:", weights.shape)
print("Rows sum to 1:",
      torch.allclose(weights.sum(dim=-1),
                     torch.ones_like(weights.sum(dim=-1))))
print("Future tokens masked:",
      torch.all(weights.triu(diagonal=1) == 0).item())


Output shape: torch.Size([2, 2, 4, 8])
Weights shape: torch.Size([2, 2, 4, 4])
Rows sum to 1: True
Future tokens masked: True


In [64]:

def causal_loss(logits, tokens):
    """logits at t predict token t+1; all sequences are equal length."""
    import torch.nn.functional as F

    # Step 1: Remove the last prediction
    shifted_logits = logits[:, :-1, :]

    # Step 2: Remove the first token from targets
    shifted_tokens = tokens[:, 1:]

    # Step 3: Calculate cross-entropy loss
    loss = F.cross_entropy(
        shifted_logits.reshape(-1, shifted_logits.size(-1)),
        shifted_tokens.reshape(-1)
    )

    return loss


In [65]:

import torch

torch.manual_seed(42)

# Batch size = 2, sequence length = 5, vocabulary size = 10
logits = torch.randn(2, 5, 10)
tokens = torch.randint(0, 10, (2, 5))

loss = causal_loss(logits, tokens)

print("Causal Loss:", loss.item())
print("Loss is finite:", torch.isfinite(loss).item())
print("Loss is nonnegative:", (loss >= 0).item())


Causal Loss: 2.679393768310547
Loss is finite: True
Loss is nonnegative: True


In [66]:

def masked_loss(logits, targets, selected):
    """selected is a Boolean B,T mask. No loss on unselected positions."""
    import torch.nn.functional as F

    # Step 1: Select logits at masked positions
    selected_logits = logits[selected]

    # Step 2: Select corresponding target tokens
    selected_targets = targets[selected]

    # Step 3: Calculate loss only for selected positions
    loss = F.cross_entropy(
        selected_logits,
        selected_targets
    )

    return loss


In [67]:

import torch

torch.manual_seed(42)

# Batch size = 2, sequence length = 5, vocabulary size = 10
logits = torch.randn(2, 5, 10)
targets = torch.randint(0, 10, (2, 5))

selected = torch.tensor([
    [False, True, False, True, False],
    [True, False, False, True, False]
])

loss = masked_loss(logits, targets, selected)

print("Masked Loss:", loss.item())
print("Loss is finite:", torch.isfinite(loss).item())
print("Loss is nonnegative:", (loss >= 0).item())
print("Selected positions:", selected.sum().item())


Masked Loss: 2.6978373527526855
Loss is finite: True
Loss is nonnegative: True
Selected positions: 4


In [68]:
class TinyTransformer(nn.Module):
    def __init__(self,vocab=12,width=24,heads=3):
        super().__init__(); self.heads=heads; self.width=width
        self.embed=nn.Embedding(vocab,width); self.pos=nn.Embedding(12,width)
        self.qkv=nn.Linear(width,3*width); self.proj=nn.Linear(width,width)
        self.norm1=nn.LayerNorm(width); self.norm2=nn.LayerNorm(width)
        self.ff=nn.Sequential(nn.Linear(width,48),nn.ReLU(),nn.Linear(48,width))
        self.head=nn.Linear(width,vocab)
    def forward(self,ids,causal=True):
        b,t=ids.shape; d=self.width//self.heads
        x=self.embed(ids)+self.pos(torch.arange(t,device=ids.device))
        q,k,v=self.qkv(self.norm1(x)).chunk(3,-1)
        q,k,v=[z.view(b,t,self.heads,d).transpose(1,2) for z in (q,k,v)]
        z,_=attention(q,k,v,causal)
        x=x+self.proj(z.transpose(1,2).contiguous().view(b,t,self.width))
        x=x+self.ff(self.norm2(x)); return self.head(x)


In [69]:

class LoRALinear(nn.Module):
    def __init__(self, base, rank=2, alpha=2):
        super().__init__()
        self.base = base
        self.scale = alpha / rank

        # Freeze the original base layer
        for p in base.parameters():
            p.requires_grad = False

        # Initialize LoRA matrices
        self.A = nn.Parameter(
            torch.randn(rank, base.in_features) * .02
        )
        self.B = nn.Parameter(
            torch.zeros(base.out_features, rank)
        )

    def forward(self, x):
        # Original frozen linear layer
        base_output = self.base(x)

        # Low-rank adaptation
        lora_output = (x @ self.A.T) @ self.B.T

        # Combine original output with scaled adaptation
        return base_output + self.scale * lora_output


In [70]:

import torch
import torch.nn as nn

torch.manual_seed(42)

base = nn.Linear(16, 10)
lora = LoRALinear(base, rank=2, alpha=2)

x = torch.randn(3, 16)

output = lora(x)

print("Output shape:", output.shape)

print("Base weights frozen:",
      all(not p.requires_grad
          for p in lora.base.parameters()))

print("LoRA matrices trainable:",
      lora.A.requires_grad and lora.B.requires_grad)

print("Initial output matches base:",
      torch.allclose(output, base(x)))

print("Trainable parameters:",
      sum(p.numel() for p in lora.parameters()
          if p.requires_grad))


Output shape: torch.Size([3, 10])
Base weights frozen: True
LoRA matrices trainable: True
Initial output matches base: True
Trainable parameters: 52


In [71]:
def data(n,seed,shift=0):
    g=torch.Generator().manual_seed(seed)
    a=torch.randint(1,5,(n,),generator=g); b=torch.randint(5,9,(n,),generator=g)
    # 0 BOS, 9 separator, 10 EOS, 11 MASK; deterministic repeated structure.
    return torch.stack([a*0,a,b,a*0+9,((a-1+shift)%4)+1,b,a*0+10],1)


In [72]:
def checks():
    torch.manual_seed(17)
    q=torch.randn(2,3,4,8); k=torch.randn_like(q); v=torch.randn_like(q)
    z,w=attention(q,k,v,True)
    assert torch.allclose(w.sum(-1),torch.ones_like(w.sum(-1)))
    assert torch.count_nonzero(w.triu(1))==0
    changed=v.clone(); changed[:,:,3]+=100
    assert torch.allclose(attention(q,k,changed,True)[0][:,:,:3],z[:,:,:3])
    base=nn.Linear(4,3); layer=LoRALinear(base)
    x=torch.randn(2,4); assert torch.allclose(layer(x),base(x))
    logits=torch.zeros(1,3,12); selected=torch.tensor([[False,True,False]])
    assert torch.isclose(masked_loss(logits,torch.tensor([[0,1,2]]),selected),torch.tensor(math.log(12)))


In [73]:
def experiment():
    torch.manual_seed(17); train=data(128,17); dev=data(64,18); test=data(64,19)
    # Samples are independently generated draws of a finite grammar, not a claim of novel grammar generalization.
    model=TinyTransformer(); opt=torch.optim.Adam(model.parameters(),lr=.01)
    initial=causal_loss(model(dev),dev).item()
    for _ in range(100):
        opt.zero_grad(); loss=causal_loss(model(train),train); loss.backward(); opt.step()
    model.eval(); base_test=causal_loss(model(test),test).item()
    adapted=data(64,20,shift=1)
    for p in model.parameters(): p.requires_grad=False
    model.head=LoRALinear(model.head)
    params=[p for p in model.parameters() if p.requires_grad]
    opt=torch.optim.Adam(params,lr=.03)
    adapted_test=data(64,21,shift=1)
    before=causal_loss(model(adapted_test),adapted_test).item()
    for _ in range(80):
        opt.zero_grad(); loss=causal_loss(model(adapted),adapted); loss.backward(); opt.step()
    after=causal_loss(model(adapted_test),adapted_test).item()
    mlm=TinyTransformer(); opt=torch.optim.Adam(mlm.parameters(),lr=.01)
    selected=torch.zeros_like(train,dtype=torch.bool); selected[:,4]=True
    corrupt=train.clone(); corrupt[selected]=11
    for _ in range(80):
        opt.zero_grad(); loss=masked_loss(mlm(corrupt,False),train,selected); loss.backward(); opt.step()
    ts=torch.zeros_like(test,dtype=torch.bool); ts[:,4]=True; tc=test.clone(); tc[ts]=11
    global trained_mlm
    trained_mlm = mlm
    trained_mlm.eval()
    return {'initial_dev_causal_loss':initial,'base_test_causal_loss':base_test,'adapter_trainable_parameters':sum(p.numel() for p in params),'adapted_distribution_loss_before':before,'adapted_test_loss_after':after,'masked_test_loss':masked_loss(mlm(tc,False),test,ts).item(),'warning':'Causal and masked losses score different events; do not rank their model quality from these numbers.'}


## Run checks and the supplied experiment

Complete the exercise functions first. A `NotImplementedError` here is expected in an untouched starter. The result dictionary covers the supplied baseline; the handout also requires comparisons and analysis that you must add below.


In [74]:
import time

checks()
_started = time.perf_counter()
results = experiment()
baseline_runtime_seconds = time.perf_counter() - _started
results


{'initial_dev_causal_loss': 2.591270923614502,
 'base_test_causal_loss': 0.46535730361938477,
 'adapter_trainable_parameters': 72,
 'adapted_distribution_loss_before': 2.428076982498169,
 'adapted_test_loss_after': 0.5069183707237244,
 'masked_test_loss': 0.0,
 'warning': 'Causal and masked losses score different events; do not rank their model quality from these numbers.'}

## Your required comparisons

Add the additional experiments, checks, plots/tables, measured runtimes, and clearly labeled results required by the handout. Add their JSON-compatible summaries to `additional_results`.


In [75]:

import time
import json
import torch

start = time.perf_counter()

# Recreate the fixed test set
test = data(64, 19)

# Select position 4 for masked prediction
selected = torch.zeros_like(test, dtype=torch.bool)
selected[:, 4] = True

original = test.clone()
original[selected] = 11

# Change a token to the right of position 4
perturbed = original.clone()
perturbed[:, 5] = (perturbed[:, 5] + 1) % 11

with torch.no_grad():
    original_logits = trained_mlm(original, False)
    changed_logits = trained_mlm(perturbed, False)

    difference = (
        original_logits[:, 4, :] -
        changed_logits[:, 4, :]
    ).abs().max().item()

runtime = time.perf_counter() - start

additional_results = {
    "right_context_max_logit_difference": float(difference),
    "right_context_visible": bool(difference > 1e-6),
    "right_context_comparison_runtime_seconds": float(runtime)
}

print(json.dumps(additional_results, indent=4))



{
    "right_context_max_logit_difference": 0.23754644393920898,
    "right_context_visible": true,
    "right_context_comparison_runtime_seconds": 0.006024258000252303
}


## Brief analysis

1. Which causal-mask bug could create deceptively low training loss?

2. What can adaptation of only an output head change, and what can it not directly change in the frozen backbone?

3. The finite grammar repeats possible sequences across independent draws. Explain why these results measure a small distribution-fitting exercise rather than held-out linguistic compositionality.

Write approximately 2–4 sentences for each question in the next cell, tied to your measured results.

1. _Your response:_A causal-mask bug can allow the model to access future tokens while predicting the next token. This makes prediction easier because the model can see information that should be hidden. As a result, the training loss may become very low without the model learning proper next-token prediction.



2. _Your response:_Adapting only the output head changes how the model maps its existing hidden representations to token predictions. In this experiment, LoRA used 72 trainable parameters and reduced the adapted test loss from 2.4281 to 0.5069. However, the adapter cannot directly update the frozen backbone's attention weights or internal representations.

3. _Your response:_The experiment uses synthetic sequences generated from a small, finite grammar, so similar or identical sequences may appear in both training and testing. The causal test loss of 0.4654 shows that the model learned to predict tokens from this generated distribution. However, these results do not demonstrate that the model can generalize to new linguistic structures or understand more complex language.

### Additional Experiment — Right-Context Influence

I changed a token to the right of the selected masked position and measured the resulting difference in the model's output logits. The maximum absolute logit difference was 0.2375, confirming that the right context influenced the masked position's predictions. The comparison completed in under 0.02 seconds.


## Save and submit

From Colab: **Share → General access → Anyone with the link → Viewer**. Click **Copy link**. Paste the link into the Canvas assignment submission.
